In [1]:
import os
import os.path
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
import os
import sys
from pathlib import Path

# === CONFIGURATION ===
# Choose which dataset to run on: "val" or "test"
DATASET_MODE = "test"  # Change to "test" for final submission

# Set to True to rebuild indices from CSV (required on first run)
# Set to False to load cached indices (faster for subsequent runs)
FORCE_REBUILD_INDICES = False

# Detect environment
KAGGLE_ENV = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if KAGGLE_ENV:
    # Kaggle paths
    DATA_PATH = Path("/kaggle/input/omnilex-data")
    MODEL_PATH = Path("/kaggle/input/llama-model")
    OUTPUT_PATH = Path("/kaggle/working")
    INDEX_PATH = Path("/kaggle/input/omnilex-indices")
    sys.path.insert(0, "/kaggle/input/omnilex-utils")
else:
    # Local development paths
    REPO_ROOT = Path(".").resolve().parent
    DATA_PATH = REPO_ROOT / "data"
    MODEL_PATH = REPO_ROOT / "models"
    OUTPUT_PATH = REPO_ROOT / "output"
    INDEX_PATH = REPO_ROOT / "data" / "processed"

# CSV corpus files for index building
LAWS_CSV = DATA_PATH / "laws_de.csv"
COURTS_CSV = DATA_PATH / "court_considerations.csv"

# Index cache paths
LAWS_INDEX_PATH = INDEX_PATH / "laws_index.pkl"
COURTS_INDEX_PATH = INDEX_PATH / "courts_index.pkl"

# Derived paths based on DATASET_MODE
QUERY_FILE = DATA_PATH / f"{DATASET_MODE}.csv"
IS_VALIDATION_MODE = DATASET_MODE == "val"

# Create output directory
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
INDEX_PATH.mkdir(parents=True, exist_ok=True)

print(f"Environment: {'Kaggle' if KAGGLE_ENV else 'Local'}")
print(f"Dataset mode: {DATASET_MODE}")
print(f"Query file: {QUERY_FILE}")
print(f"Validation mode: {IS_VALIDATION_MODE}")
print(f"Force rebuild indices: {FORCE_REBUILD_INDICES}")
print(f"\nCorpus files:")
print(f"  Laws CSV: {LAWS_CSV} ({LAWS_CSV.stat().st_size / 1e6:.1f} MB)" if LAWS_CSV.exists() else f"  Laws CSV: {LAWS_CSV} (NOT FOUND)")
print(f"  Courts CSV: {COURTS_CSV} ({COURTS_CSV.stat().st_size / 1e9:.2f} GB)" if COURTS_CSV.exists() else f"  Courts CSV: {COURTS_CSV} (NOT FOUND)")
print(f"\nIndex cache: {INDEX_PATH}")

Environment: Local
Dataset mode: test
Query file: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/test.csv
Validation mode: False
Force rebuild indices: False

Corpus files:
  Laws CSV: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/laws_de.csv (73.0 MB)
  Courts CSV: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/court_considerations.csv (2.43 GB)

Index cache: /root/autodl-tmp/llm-legal/Omnilex-Agentic-Retrieval-Competition/data/processed


# 2. Load Corpora and Build/Load Indices

In [3]:
import re

In [4]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

dense_model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True)
reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True) # Setting use_fp16 to True speeds up computation with a slight performance degradation

In [5]:
import query_by_dense
import citation_utils

court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = {}
for citation, text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()):
    # if citation in court_consideration_d:
    #     court_consideration_d[citation] = court_consideration_d[citation] + '\n\n' + text
    # else:
    #     court_consideration_d[citation] = text
    court_consideration_d[citation] = text

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

test_df = pd.read_csv('../data/test_rewrite_001.csv')
id_l = []
citation_l = []

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]

print("data loaded")

data loaded


In [6]:
import dense_index
from dense_index import DenseIndex

print(dense_model.normalize_embeddings)
court_dense_index = DenseIndex(dense_model, "../data/processed/_dense_court", court_doc)
court_dense_index.info()

True
DenseIndex.embeddings:  (2776718, 1024)
[dense_index] documents.len: 2476315 parent_idx.len: 2776718


In [7]:
from sparse_index import SparseIndex

court_sparse_index = SparseIndex(dense_model, "../data/processed/_sparse_court", court_doc)
court_sparse_index.load()



In [8]:
import citation_utils
import rerank_utils

id_l = []
for id, q, q_en in tqdm(zip(test_df['query_id'].tolist(), test_df['query'].tolist(), test_df['query_en'].tolist()), total=len(test_df)):
    print("query len:", len(q))
    id_l.append(id)
    citations = []

    first_layer_citation = []
    for citation in citation_utils.extract_citations_from_text(q_en):
        first_layer_citation.append(citation)


    # test_results = courts_index.search(q, top_k=1000)[0]['hits']
    # test_results = []
    # if len(first_layer_citation) > 0:
    #     print("====> sparse search")
    #     test_results_sparse = court_sparse_index.search(q, 1000)
    #     test_results.extend(test_results_sparse)
    # else:
    #     print("====> dense search")
    #     test_results_dense = court_dense_index.search(q, 1000)
    #     test_results.extend(test_results_dense)

    test_results_0 = []
    if len(first_layer_citation) > 0:
        print("====> sparse search")
        test_results_sparse = court_sparse_index.search(q, 100)
        test_results_0.extend(test_results_sparse)
    else:
        print("====> dense search")
        test_results_dense = court_dense_index.search(q, 100)
        test_results_0.extend(test_results_dense)

    test_results = []
    
    for hit in test_results_0:
        test_results.append(hit)
        test_results.extend(court_dense_index.search(hit['text'], 10))
    
    _set = set([hit['citation'] for hit in test_results])
    first_layer_citation.extend(list(_set))

    print("first_layer_citation.len:", len(first_layer_citation))

    raw_hits = citation_utils.BFS_citation(court_consideration_d, law_d, first_layer_citation, max_level=3) # 广度优先搜索

    print("raw_hits.len:", len(raw_hits))
    
    court_hits = [hits for hits in raw_hits if hits['citation'] in court_consideration_d]
    law_hits = [hits for hits in raw_hits if hits['citation'] in law_d]

    # court_l = rerank_utils.rerank_by_dense_batch(reranker, q, court_hits, 20, 20)
    court_l = rerank_utils.rerank_by_dense_batch_chunked(reranker, q, court_hits, 20, 20, 384, 128)
    # court_l = query_by_dense.query_by_dense(model, q, court_hits, 20)
    for court in court_l:
        citations.append(court['citation'])

    # law_l = rerank_utils.rerank_by_dense_batch(reranker, q, law_hits, 20, 20)
    law_l = rerank_utils.rerank_by_dense_batch_chunked(reranker, q, law_hits, 20, 20, 384, 128)
    # law_l = query_by_dense.query_by_dense(model, q, law_hits, 20)
    for law in law_l:
        citations.append(law['citation'])

    citations = list(set(citations))
    citation_l.append(';'.join(citations))
    print(id)

result_df = pd.DataFrame({'query_id':id_l, 'predicted_citations':citation_l})
result_df.to_csv("../data/result.csv", index=False)

  0%|          | 0/40 [00:00<?, ?it/s]

query len: 394
====> dense search


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


first_layer_citation.len: 637
raw_hits.len: 756



rerank_by_dense:   0%|          | 0/669 [00:00<?, ?it/s]You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.

rerank_by_dense: 100%|██████████| 669/669 [00:14<00:00, 44.98it/s]

  2%|▎         | 1/40 [01:38<1:03:46, 98.12s/it]

...

test_038
query len: 561
====> dense search
first_layer_citation.len: 779
raw_hits.len: 1100



rerank_by_dense: 100%|██████████| 912/912 [00:17<00:00, 51.40it/s]

 98%|█████████▊| 39/40 [1:05:26<01:41, 101.15s/it]

test_039
query len: 271
====> sparse search
first_layer_citation.len: 361
raw_hits.len: 446



rerank_by_dense: 100%|██████████| 404/404 [00:07<00:00, 52.55it/s]

100%|██████████| 40/40 [1:07:00<00:00, 100.52s/it]

test_040
